# Day 2 · Unit 1.2 — Tokens, Context and Latency
*100 Days of AI Engineering (CampusX) · notes + experiments, in simple words*

**Mental model:** `Your text → tokens → (you pay per token, wait per token, and fit everything inside one token limit)`

**Goal of today:** estimate what a request will **cost**, roughly how long it will **take**, and decide whether a task needs **more context** or **better-chosen context**.

**How to use this notebook:** read Part 1, then run the experiments top to bottom. Experiments 1–2 need no API key. Experiments 3–6 need the Groq key from Colab Secrets (same as Day 1). After running, check the "What to look for" box under each experiment.

## Part 1 — The 7 ideas in simple words

### 1. Tokens, not words
Models do not read words or letters. They read **tokens**: small chunks of text (a whole word, part of a word, a symbol, a space + word).
- Rough rule for English: **1 token ≈ 4 characters ≈ ¾ of a word**. This is only a rough guide, so always count for real.
- **Different models cut text differently.** GPT models, Qwen, Llama etc. each have their own tokenizer, so the *same sentence* can be 20 tokens for one model and 26 for another.
- You are **billed in tokens**, and the model's limit is in tokens too.

### 2. Why some text costs more
Tokenizers are built from training text that is mostly English. So:
- Common English words → often **1 token**.
- Hindi, Urdu, Hinglish, code and numbers get chopped into **many small pieces** (sometimes even pieces of a single character).
- Same meaning, **more tokens** → higher cost, slower reply, and less room left in the context window.

### 3. Input vs output tokens
- **Input tokens** = everything you send. **Output tokens** = everything the model writes back.
- You pay for **both**. Output is usually **priced higher**.
- Why output is slower: the model writes **one token at a time** (each new token needs the previous ones, remember Unit 1.1). Your input is read much faster, in one go.

### 4. Latency you can feel
- **Time to first token (TTFT):** how long until the answer *starts*. Includes network, queueing, reading your input, and (for reasoning models) hidden thinking.
- **Generation speed:** tokens per second after it starts.
- **Rough formula:** `total time ≈ TTFT + output_tokens ÷ tokens_per_second`
- So a **long answer takes longer than a long question**. To make an app feel fast: ask for shorter output, stream the answer, use a smaller or faster model.

### 5. What fills the context window
The **context window** is the model's working space, measured in tokens. **All of this shares one limit:**
instructions (system prompt) + conversation history + documents you paste + examples + tool results + **the model's own answer**.

Important: the API is **stateless**. It does not remember the last call, so you **resend the whole history every turn**. Long chats get more expensive each turn.

### 6. Context is not knowledge, and not memory
| Thing | What it is | Lasts how long |
|---|---|---|
| **Context** | Text in the window right now | Only for this request |
| **Knowledge** | What the model learned in training | Fixed (has a cutoff date) |
| **Memory** | Remembering you across chats | Does not exist by default. **You must build it** (save notes in a database and put them back into the context) |

### 7. Bigger windows are not better windows
A 1-million-token window does not mean you should fill it.
- Facts buried **in the middle** of a very long input are often missed ("lost in the middle").
- Extra irrelevant text **dilutes focus** and adds cost and delay.
- So the real skill is **better-chosen context**: send only the relevant parts (this leads to RAG later in the course).

## Setup

In [ ]:
!pip -q install tiktoken transformers openai pandas

In [ ]:
import time, random
import pandas as pd
import tiktoken
from openai import OpenAI

# Groq client (only needed for Experiments 3-6). The key comes from Colab Secrets, never typed here.
try:
    from google.colab import userdata
    client = OpenAI(api_key=userdata.get("GROQ_API_KEY"), base_url="https://api.groq.com/openai/v1")
    MODEL = "openai/gpt-oss-20b"      # a model that worked for you on Day 1
    print("Groq client ready:", MODEL)
except Exception as e:
    client = None
    print("No Groq client (Experiments 1-2 still work):", e)

## Experiment 1 — Count tokens: English vs Hindi vs code vs numbers
*(This is the official "Try it" for today.)*

**What this code does**
1. Makes 6 samples: an English paragraph, its Hindi translation, the same thing in Hinglish (Roman letters), Urdu, a small piece of Python code, and a list of numbers.
2. Counts tokens with **four different tokenizers** (two OpenAI ones through `tiktoken`, and GPT-2 and Qwen through Hugging Face).
3. Shows a table with characters, tokens and **characters per token** (higher = cheaper text).

In [ ]:
english  = ("Artificial intelligence is changing the way we work, learn and build software. "
            "Companies of every size are now using language models to answer customer questions, "
            "summarize documents and write code.")

hindi    = ("आर्टिफिशियल इंटेलिजेंस हमारे काम करने, सीखने और सॉफ्टवेयर बनाने के तरीके को बदल रहा है। "
            "हर आकार की कंपनियां अब ग्राहकों के सवालों के जवाब देने, दस्तावेज़ों का सारांश बनाने "
            "और कोड लिखने के लिए भाषा मॉडल का उपयोग कर रही हैं।")

hinglish = ("AI hamare kaam karne, seekhne aur software banane ka tareeka badal raha hai. "
            "Aaj har size ki companies customers ke sawalon ke jawab dene, documents ka summary banane "
            "aur code likhne ke liye language models use kar rahi hain.")

urdu     = ("مصنوعی ذہانت ہمارے کام کرنے، سیکھنے اور سافٹ ویئر بنانے کے طریقے کو بدل رہی ہے۔ "
            "ہر سائز کی کمپنیاں اب صارفین کے سوالوں کے جواب دینے، دستاویزات کا خلاصہ بنانے "
            "اور کوڈ لکھنے کے لیے لینگویج ماڈلز استعمال کر رہی ہیں۔")

code_snippet = '''def total_price(items, tax_rate=0.18):
    subtotal = sum(item["price"] * item["qty"] for item in items)
    return round(subtotal * (1 + tax_rate), 2)
'''

random.seed(42)
numbers = ", ".join(str(random.randint(1, 10 ** random.randint(1, 8))) for _ in range(40))

samples = {
    "English": english,
    "Hindi": hindi,
    "Hinglish (Roman)": hinglish,
    "Urdu": urdu,
    "Python code": code_snippet,
    "Numbers": numbers,
}
print(numbers)

In [ ]:
from transformers import AutoTokenizer

enc_o200k  = tiktoken.get_encoding("o200k_base")    # newer OpenAI tokenizer
enc_cl100k = tiktoken.get_encoding("cl100k_base")   # older OpenAI tokenizer

hf_tokenizers = {}
for name in ["gpt2", "Qwen/Qwen2.5-0.5B"]:
    try:
        hf_tokenizers[name] = AutoTokenizer.from_pretrained(name)
    except Exception as e:
        print("skipped", name, "->", e)

def count(text):
    row = {"chars": len(text),
           "o200k": len(enc_o200k.encode(text)),
           "cl100k": len(enc_cl100k.encode(text))}
    for name, tok in hf_tokenizers.items():
        row[name] = len(tok.encode(text, add_special_tokens=False))
    return row

df = pd.DataFrame({k: count(v) for k, v in samples.items()}).T
df["chars per token (o200k)"] = (df["chars"] / df["o200k"]).round(2)
# Only meaningful for the translations (same meaning as English):
df["x English (o200k)"] = (df["o200k"] / df.loc["English", "o200k"]).round(2)
df

### What to look for
- **Chars per token:** English should be the highest (cheapest). Hindi, Urdu, code and numbers should be lower, meaning more tokens for the same length.
- **x English column:** only valid for Hindi, Hinglish and Urdu, because they say the same thing as the English paragraph. A value of 3 means the same meaning costs 3 times more tokens.
- **Compare columns:** the same text gets *different* counts across tokenizers. Which tokenizer is kindest to Hindi?
- **Numbers:** notice each number is split into chunks of digits, so long numbers cost a lot.
- **Hinglish vs Hindi:** is Roman-letter Hinglish cheaper than Devanagari? This matters for your Hindi-English apps.

### See *how* the text is split (this is the part that explains the cost)

In [ ]:
def split_view(text, enc, limit=20):
    ids = enc.encode(text)[:limit]
    return [enc.decode_single_token_bytes(t) for t in ids]

print("English  (o200k):", split_view(english, enc_o200k))
print()
print("Hindi    (o200k):", split_view(hindi, enc_o200k))
print()
print("Hindi   (cl100k):", split_view(hindi, enc_cl100k))
print()
print("Same English sentence, 3 tokenizers:")
print(" o200k :", split_view(english, enc_o200k, 12))
print(" cl100k:", split_view(english, enc_cl100k, 12))
if "gpt2" in hf_tokenizers:
    ids = hf_tokenizers["gpt2"].encode(english, add_special_tokens=False)[:12]
    print(" gpt2  :", hf_tokenizers["gpt2"].convert_ids_to_tokens(ids))

### What to look for
- English tokens look like whole words (`b'Art'`, `b' intelligence'`).
- Hindi shows things like `b'\xe0\xa4\x86'`. These are **raw bytes**: one Hindi character is several bytes, and the tokenizer often has no big token for it, so it falls back to tiny pieces. That is exactly why Hindi costs more.
- The same English sentence is cut slightly differently by each tokenizer.

## Experiment 2 — Estimate cost if I send each text 10,000 times

**Formula:** `cost = calls × (input_tokens × input_price + output_tokens × output_price) ÷ 1,000,000`
(prices are per **million** tokens)

> ⚠️ The prices below are **example numbers only**. Replace them with the current prices for your model from your provider's pricing page.

In [ ]:
INPUT_PRICE_PER_M  = 0.15   # EXAMPLE price, USD per 1M input tokens  -> replace with real price
OUTPUT_PRICE_PER_M = 0.60   # EXAMPLE price, USD per 1M output tokens -> replace with real price
CALLS = 10_000
ASSUMED_OUTPUT_TOKENS = 200 # pretend every reply is about 200 tokens

def cost_usd(in_tok, out_tok, calls=CALLS):
    return calls * (in_tok * INPUT_PRICE_PER_M + out_tok * OUTPUT_PRICE_PER_M) / 1_000_000

rows = []
for name in samples:
    n = int(df.loc[name, "o200k"])
    rows.append({
        "text": name,
        "input tokens": n,
        f"input-only cost x{CALLS:,} ($)": round(cost_usd(n, 0), 4),
        "input + 200 output tokens ($)": round(cost_usd(n, ASSUMED_OUTPUT_TOKENS), 4),
    })
pd.DataFrame(rows)

### What to look for
- The input-only column is the answer to the official "Try it". Which text is the most expensive to send 10,000 times?
- Our samples are tiny, so costs look like cents. **Real apps send thousands of tokens per call.** Try the next cell.
- Look at the second column: once you add output, the **output part often dominates** because it costs more per token.

In [ ]:
# A realistic RAG-style request: system prompt + retrieved document + question
system_tokens, doc_tokens, question_tokens, answer_tokens = 500, 3000, 50, 300
in_tok = system_tokens + doc_tokens + question_tokens
print(f"One request: {in_tok} input + {answer_tokens} output tokens")
print(f"Cost for {CALLS:,} requests: ${cost_usd(in_tok, answer_tokens):,.2f}")

# Same app, but the user writes in Hindi and everything costs ~3x tokens (use your own ratio from Experiment 1)
ratio = 3
print(f"If tokens were {ratio}x higher: ${cost_usd(in_tok * ratio, answer_tokens * ratio):,.2f}")

### Chat history makes each turn more expensive
Because the API is stateless, every turn you resend **system prompt + the full history + the new message**.

In [ ]:
SYSTEM, USER_MSG, REPLY = 500, 100, 200     # tokens (assumptions)
history = 0
total_input = 0
print("turn | input tokens sent this turn | total input tokens so far")
for turn in range(1, 11):
    sent = SYSTEM + history + USER_MSG
    total_input += sent
    history += USER_MSG + REPLY
    print(f"{turn:>4} | {sent:>27} | {total_input:>25}")

### What to look for
Input per turn **keeps growing**. The *total* over 10 turns is far more than 10 × the first turn. That is why long chats get expensive, and why apps trim or summarize old messages.

## Experiment 3 — Measure latency: time to first token and speed
*(Needs the Groq client.)*

**What this code does**
Calls the model with **streaming**, so we can note when the first visible text arrives (**TTFT**) and when it finishes. We test three prompts:
1. **Short answer** (one sentence)
2. **Long answer** (about 300 words)
3. **Long question, tiny answer** (about 3,000 tokens in, one word out)

*Notes: `out_tokens≈` is counted by re-tokenizing the reply, so it is approximate. `gpt-oss` is a reasoning model, so its hidden thinking time is included in TTFT.*

In [ ]:
def timed_call(prompt, max_tokens=1200):
    t0 = time.perf_counter()
    first = None
    text = ""
    stream = client.chat.completions.create(
        model=MODEL, stream=True, reasoning_effort="low",
        max_completion_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
    )
    for chunk in stream:
        if not chunk.choices:
            continue
        piece = chunk.choices[0].delta.content
        if piece:
            if first is None:
                first = time.perf_counter() - t0
            text += piece
    total = time.perf_counter() - t0
    first = first if first is not None else total
    n_out = len(enc_o200k.encode(text))
    return {"TTFT (s)": round(first, 2), "total (s)": round(total, 2),
            "out_tokens≈": n_out, "tokens/sec≈": round(n_out / max(total - first, 1e-9), 1)}

long_input = (english + " ") * 80 + "\n\nReply with only the word OK."
tests = {
    "short answer":            "In one sentence, what is a token in an LLM?",
    "long answer":             "Explain what tokens are in LLMs in about 300 words.",
    "long question, 1-word answer": long_input,
}

if client is None:
    print("No Groq client. Run the setup cell with your key in Colab Secrets.")
else:
    results = {}
    for name, p in tests.items():
        try:
            r = timed_call(p)
            r["input_tokens≈"] = len(enc_o200k.encode(p))
            results[name] = r
        except Exception as e:
            results[name] = {"error": str(e)[:120]}
    display(pd.DataFrame(results).T)

### What to look for
- **Long answer** should take clearly longer in total than the short one, because output is generated token by token.
- **Long question, 1-word answer** has thousands of input tokens but should finish **much faster than the long answer**. Long questions are cheap in time; long answers are not.
- TTFT may be similar across tests (or higher for the reasoning model). Speeds on a free tier can vary from run to run, so run it twice.
- Use your numbers in the formula: `total ≈ TTFT + output_tokens ÷ tokens_per_sec`.

## Experiment 4 — What fills the context window?
**What this code does:** builds a typical request out of its 6 parts, counts the tokens in each, and shows how much of the window is used and how much is left for the answer.

In [ ]:
CONTEXT_LIMIT = 128_000        # EXAMPLE limit -> check the real limit of your model
RESERVED_FOR_ANSWER = 500

parts = {
    "system instructions": "You are a helpful support agent for an online clothing store. Be polite, short and accurate. Never promise refunds.",
    "conversation history": "User: Hi, my order is late.\nAssistant: Sorry to hear that! Can you share your order number?\nUser: It is 48213.\nAssistant: Thanks, checking now.",
    "retrieved document": "Return policy: items can be returned within 14 days if unused and in original packaging. Refunds are processed in 5-7 working days. " * 20,
    "few-shot examples": "Q: Where is my parcel?\nA: Please share your order number.\nQ: Can I change my address?\nA: Yes, before the order ships.",
    "tool result": '{"order_id": 48213, "status": "in transit", "eta": "2 days"}',
    "user question": "When will my order arrive and can I return it?",
}

rows = [{"part": k, "tokens": len(enc_o200k.encode(v))} for k, v in parts.items()]
budget = pd.DataFrame(rows)
used = budget["tokens"].sum()
budget["% of window"] = (budget["tokens"] / CONTEXT_LIMIT * 100).round(3)
display(budget)
print(f"Used by input: {used} tokens | reserved for answer: {RESERVED_FOR_ANSWER} | still free: {CONTEXT_LIMIT - used - RESERVED_FOR_ANSWER}")

### What to look for
- The **retrieved document** is by far the biggest part. In real apps, documents and history are what fill the window.
- The model's **own answer** also needs room. If input uses the whole window, the answer gets cut off.
- Try changing `* 20` to `* 2000` and see how fast the window fills.

## Experiment 5 — Context is not memory
**What this code does:** asks "What is my name?" twice. Once with **no history**, once with the earlier messages **included in the context**.

In [ ]:
def chat(messages, max_tokens=400):
    r = client.chat.completions.create(
        model=MODEL, reasoning_effort="low",
        max_completion_tokens=max_tokens, messages=messages)
    return (r.choices[0].message.content or "[empty]").strip()

if client is None:
    print("No Groq client.")
else:
    print("WITHOUT history:\n", chat([{"role": "user", "content": "What is my name?"}]))
    print()
    with_history = [
        {"role": "user", "content": "My name is Ali."},
        {"role": "assistant", "content": "Nice to meet you, Ali!"},
        {"role": "user", "content": "What is my name?"},
    ]
    print("WITH history:\n", chat(with_history))

### What to look for
- Without history, the model cannot know your name. A new API call starts empty.
- With history, it answers correctly, but **only because you resent the text**. That is context, not memory.
- Real "memory" means saving facts (database, file) and **inserting them back** into the context on later calls.

## Experiment 6 — Do long inputs hide facts? (mini needle-in-a-haystack)
**What this code does:** hides one secret sentence (the "needle") at the **start**, **middle** and **end** of a long boring text, then asks the model to find it. This checks whether *position* and *length* matter.

*Start with the default size. To push harder, raise `n_sentences` (for example 1500 ≈ 30k tokens). On a free tier you may hit rate limits. If so, lower it.*

In [ ]:
NEEDLE = "The secret code for the Karachi project is BLUE-7421."

def make_haystack(n_sentences, needle_pos):
    sents = [f"Log {i}: warehouse {i % 17} handled {(i * 37) % 1000} parcels on day {i % 28 + 1}."
             for i in range(n_sentences)]
    sents.insert(int(needle_pos * n_sentences), NEEDLE)
    return " ".join(sents)

def needle_test(n_sentences=200):
    rows = []
    for pos in [0.0, 0.5, 1.0]:
        question = (make_haystack(n_sentences, pos) +
                    "\n\nQuestion: What is the secret code for the Karachi project? Answer with only the code.")
        try:
            ans = chat([{"role": "user", "content": question}], max_tokens=500)
        except Exception as e:
            ans = "ERROR: " + str(e)[:80]
        rows.append({"needle position": {0.0: "start", 0.5: "middle", 1.0: "end"}[pos],
                     "input tokens≈": len(enc_o200k.encode(question)),
                     "found?": "BLUE-7421" in ans,
                     "model said": ans[:60]})
    return pd.DataFrame(rows)

if client is None:
    print("No Groq client.")
else:
    display(needle_test(200))

### What to look for
- At small sizes a good model often finds the needle everywhere. That is fine. **It does not prove long inputs are safe.**
- Increase `n_sentences` and re-run. Watch whether the **middle** gets missed first, or answers become wrong.
- Even if it finds the fact, the long input still costs more and takes longer. A smaller, well-chosen input is better.
- Lesson: **test with your own data at your real input size**. Do not trust the advertised window size.

## Self-check questions and answers

**1. Why can the same sentence have different token counts in different models?**
Each model family has its own tokenizer, trained on different data with a different vocabulary, so it cuts text into different pieces. So you should count tokens with the tokenizer of the model you actually use.

**2. Why does Hindi (or Urdu, code, numbers) often use more tokens than English?**
Tokenizers are built mostly from English-heavy text, so English words get big single tokens. Other scripts, symbols and digits get split into many small pieces, sometimes down to single bytes. Same meaning, more tokens, so more cost, more delay and less room in the window.

**3. Do you pay for input, output, or both? Which is usually more expensive per token?**
Both. Output is usually priced higher per token, and it is also generated one token at a time.

**4. What are time to first token and generation speed? Why does a long answer take longer than a long question?**
TTFT is the wait until the first piece of the answer appears. Generation speed is tokens per second after that. Your input is processed quickly in one go, but the output is produced token by token, so total time ≈ TTFT + output_tokens ÷ speed. More output tokens means more time.

**5. What shares the context window?**
System instructions, conversation history, documents, examples, tool results, the user's question, and the model's own answer. All of them must fit inside one limit.

**6. What is the difference between context, knowledge and memory?**
Context is the temporary text in the window for this request. Knowledge is what the model learned in training (fixed, with a cutoff date). Memory across chats does not exist by default. You build it by saving information and putting it back into the context.

**7. Why is a bigger context window not automatically better?**
Facts in the middle of very long inputs are often missed, irrelevant text dilutes the model's focus, and longer inputs cost more and take more time. It is usually better to send fewer, better-chosen tokens.

---

## ✅ Day 2 Outcome

**Estimate cost:** `cost per request = input_tokens × input_price + output_tokens × output_price` (prices are per million tokens). Multiply by the number of calls. Count tokens with the right tokenizer, and remember Hindi, code and numbers cost more.

**Estimate time:** `time ≈ TTFT + output_tokens ÷ tokens_per_second`. Output length matters far more than input length.

**More context or better-chosen context?**
- If the model fails because it **lacks the information** → add the missing facts (documents, tool results).
- If the window is already **full of mostly irrelevant text**, or the answer misses details buried inside → send **less but better**: retrieve only relevant parts, trim old history, summarize, put key facts near the start or end.

**Next:** Unit 1.3 — Beyond Text: Images, Audio and PDFs.